# NLP Pipeline Model Comparison

This notebook trains and evaluates three embedding approaches:
- **TF-IDF** – classical sparse keyword vectors
- **Word2Vec** – context-aware dense word vectors (trained from scratch)
- **SBERT (Baseline)** – deep contextual sentence embeddings (`all-MiniLM-L6-v2`)

The goal is to show why SBERT outperforms traditional methods for semantic
topic clustering on the Sports / Health / Politics dataset.


In [1]:
import os
import re
import csv
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize as l2_normalize
from sklearn.metrics import (
    accuracy_score, homogeneity_score,
    completeness_score, v_measure_score
)
from sklearn.manifold import TSNE

# ── Stop-words (excluded from embedding, kept in W2V training)
STOP_WORDS = {
    'a','an','the','and','or','but','in','on','at','to','for',
    'of','with','by','from','is','are','was','were','be','been',
    'being','have','has','had','do','does','did','will','would',
    'could','should','may','might','shall','can','it','its',
    'this','that','these','those','as','if','not','no','so',
    'than','then','their','they','his','her','he','she','we',
    'our','you','your','my','me','him','us','i','up','out',
    'about','into','through','during','before','after','also',
    'just','more','other','such','only','over','any','what',
    'which','who','whom','when','where','why','how','all','each',
    'both','few','most','some','same','too','very','here','there',
    'while','since','between','under','again','further','once',
}

QUERY_DICT = {
    "Sports": [
        "football match result", "player scored a goal",
        "championship final winner", "basketball game score",
        "olympic athlete performance", "cricket world cup",
        "tennis grand slam tournament", "sports team ranking",
    ],
    "Health": [
        "vaccine dose immune response", "hospital patient treatment",
        "mental health awareness", "nutrition diet fitness",
        "disease symptoms medication", "medical research clinical trial",
        "public health policy", "doctor nurse healthcare",
    ],
    "Politics": [
        "senate vote bill congress", "election campaign result",
        "government policy reform", "president prime minister decision",
        "political party debate", "international diplomacy treaty",
        "parliament legislation", "democracy rights protest",
    ],
}

def load_data(path):
    texts, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['text'].strip():
                texts.append(row['text'].strip())
                labels.append(row['label'].strip())
    return texts, labels

def cosine_distance(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0: return 1.0
    return 1.0 - np.dot(a, b) / (na * nb)

def majority_vote_mapping(cluster_ids, train_labels):
    """Greedy 1-to-1 cluster->topic mapping using plurality vote."""
    unique_clusters = list(np.unique(cluster_ids))
    cluster_votes = {
        cid: Counter([train_labels[i] for i, c in enumerate(cluster_ids) if c == cid])
        for cid in unique_clusters
    }
    all_pairs = sorted(
        [(cid, topic, cluster_votes[cid].get(topic, 0))
         for cid in unique_clusters for topic in set(train_labels)],
        key=lambda x: -x[2]
    )
    mapping, used = {}, set()
    for cid, topic, _ in all_pairs:
        if cid not in mapping and topic not in used:
            mapping[cid] = topic; used.add(topic)
    return mapping

def plot_clusters(embeddings, labels, title):
    coords = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(embeddings)
    df = pd.DataFrame({'x': coords[:,0], 'y': coords[:,1], 'label': labels})
    palette = {'Sports': '#4CAF50', 'Health': '#2196F3', 'Politics': '#FF5722'}
    plt.figure(figsize=(8,5))
    for lbl, grp in df.groupby('label'):
        plt.scatter(grp.x, grp.y, label=lbl, alpha=0.6, s=20, color=palette.get(lbl,'gray'))
    plt.title(title); plt.legend(); plt.tight_layout()
    fname = f"cluster_plot_{title.replace(' ','_')}.png"
    plt.savefig(fname, dpi=80); plt.close()
    print(f"Saved: {fname}")

train_texts, train_labels = load_data('data/train.csv')
test_texts,  test_labels  = load_data('data/test.csv')
print(f"Train: {len(train_texts)} samples | Test: {len(test_texts)} samples")
print(f"Classes: {sorted(set(train_labels))}")


Train: 915 samples | Test: 396 samples
Classes: ['Health', 'Politics', 'Sports']


In [2]:
# TF-IDF Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

print('Training TF-IDF...')
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
train_embeddings_tfidf = vectorizer.fit_transform(train_texts).toarray()
test_embeddings_tfidf  = vectorizer.transform(test_texts).toarray()

query_vectors_tfidf = {
    topic: vectorizer.transform(phrases).toarray().mean(axis=0)
    for topic, phrases in QUERY_DICT.items()
}

# TF-IDF lives in sparse keyword space — query-vector cosine mapping works here
kmeans_tfidf = KMeans(n_clusters=3, random_state=42, n_init=20)
train_cluster_labels_tfidf = kmeans_tfidf.fit_predict(train_embeddings_tfidf)

# Build unique 1-to-1 mapping via centroid-to-query cosine distance
from sklearn.metrics import accuracy_score, homogeneity_score, completeness_score, v_measure_score
unique_clusters = list(np.unique(train_cluster_labels_tfidf))
dist_matrix = {}
for cid in unique_clusters:
    centroid = train_embeddings_tfidf[train_cluster_labels_tfidf == cid].mean(axis=0)
    dist_matrix[cid] = {t: cosine_distance(centroid, q) for t, q in query_vectors_tfidf.items()}
all_pairs = sorted(
    [(cid, t, dist_matrix[cid][t]) for cid in unique_clusters for t in dist_matrix[cid]],
    key=lambda x: x[2]
)
mapping_tfidf, used_topics = {}, set()
for cid, t, d in all_pairs:
    if cid not in mapping_tfidf and t not in used_topics:
        mapping_tfidf[cid] = t; used_topics.add(t)
print(f'TF-IDF Cluster->Topic mapping: {mapping_tfidf}')

tfidf_test_preds = [mapping_tfidf[cid] for cid in kmeans_tfidf.predict(test_embeddings_tfidf)]
res_tfidf = {
    'Accuracy':     accuracy_score(test_labels, tfidf_test_preds),
    'Homogeneity':  homogeneity_score(test_labels, tfidf_test_preds),
    'Completeness': completeness_score(test_labels, tfidf_test_preds),
    'V-Measure':    v_measure_score(test_labels, tfidf_test_preds)
}
print('TF-IDF Test Metrics:', res_tfidf)


Training TF-IDF...


TF-IDF Cluster->Topic mapping: {np.int32(1): 'Health', np.int32(0): 'Sports', np.int32(2): 'Politics'}
TF-IDF Test Metrics: {'Accuracy': 0.7752525252525253, 'Homogeneity': np.float64(0.6133131311985621), 'Completeness': np.float64(0.7122538602937147), 'V-Measure': np.float64(0.6590910124779559)}


In [3]:
# TF-IDF Embedding Inspection
print('\n--- TF-IDF Embedding Inspection ---')
print(f'Matrix Shape: {train_embeddings_tfidf.shape} (Sparse keyword counts, IDF-weighted)')
sample = train_embeddings_tfidf[0]
nonzero_idx = np.nonzero(sample)[0][:20]
print('Sample TF-IDF Vector (first 20 nonzero values):')
print(dict(zip(vectorizer.get_feature_names_out()[nonzero_idx], sample[nonzero_idx].round(4))))
plot_clusters(test_embeddings_tfidf, tfidf_test_preds, 'TF-IDF Predicted Clusters (t-SNE)')



--- TF-IDF Embedding Inspection ---
Matrix Shape: (915, 5000) (Sparse keyword counts, IDF-weighted)
Sample TF-IDF Vector (first 20 nonzero values):
{'assuming': np.float64(0.2348), 'combat': np.float64(0.4011), 'come': np.float64(0.1742), 'did': np.float64(0.151), 'does': np.float64(0.1556), 'effects': np.float64(0.2348), 'given': np.float64(0.1864), 'health': np.float64(0.1474), 'hear': np.float64(0.1836), 'impacted': np.float64(0.2348), 'just': np.float64(0.1087), 'lasting': np.float64(0.2348), 'like': np.float64(0.1168), 'mental': np.float64(0.2466), 'mess': np.float64(0.217), 'military': np.float64(0.2348), 'non': np.float64(0.1786), 'ptsd': np.float64(0.2005), 'short': np.float64(0.1894), 'term': np.float64(0.2106)}


Saved: cluster_plot_TF-IDF_Predicted_Clusters_(t-SNE).png


In [4]:
# Word2Vec Pipeline (from scratch) — 3 bug fixes applied
# Fix 1: Punctuation stripped before tokenisation
# Fix 2: Stop-words excluded from embedding step (kept in W2V training)
# Fix 3: L2-normalise + supervised nearest-centroid classification
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer as TFV
from sklearn.preprocessing import normalize

def tokenize(texts):
    return [re.sub(r'[^a-z\s]', '', t.lower()).split() for t in texts]

train_tokens = tokenize(train_texts)
test_tokens  = tokenize(test_texts)

print('Training Word2Vec (50 epochs)...')
w2v = Word2Vec(sentences=train_tokens, vector_size=100, window=7,
               min_count=1, workers=1, epochs=50, seed=42)

# IDF weights aligned to W2V vocab
vocab_list = list(w2v.wv.index_to_key)
tfidf_w2v = TFV(vocabulary={w: i for i, w in enumerate(vocab_list)})
tfidf_w2v.fit(train_texts)
idf_w2v = dict(zip(tfidf_w2v.get_feature_names_out(), tfidf_w2v.idf_))

def get_embedding(tokens):
    vecs, weights = [], []
    for t in tokens:
        if t in STOP_WORDS: continue          # Fix 2: skip stop-words
        if t in w2v.wv and t in idf_w2v:
            weight = idf_w2v[t]
            vecs.append(w2v.wv[t] * weight)
            weights.append(weight)
    if not vecs: return np.zeros(w2v.vector_size)
    return np.sum(vecs, axis=0) / np.sum(weights)

print('Building W2V embeddings...')
raw_train_w2v = np.array([get_embedding(t) for t in train_tokens], dtype=np.float64)
raw_test_w2v  = np.array([get_embedding(t) for t in test_tokens],  dtype=np.float64)

# Fix 3a: L2-normalize (cosine-equivalent KMeans)
train_embeddings_w2v = normalize(raw_train_w2v)
test_embeddings_w2v  = normalize(raw_test_w2v)

# Fix 3b: Supervised nearest-centroid (avoids random cluster-topic assignment)
print('Building per-class prototype centroids...')
unique_topics = sorted(set(train_labels))
centroids_w2v = {}
for topic in unique_topics:
    idxs = [i for i, lbl in enumerate(train_labels) if lbl == topic]
    centroids_w2v[topic] = normalize(train_embeddings_w2v[idxs].mean(axis=0, keepdims=True))[0]
    print(f"  '{topic}' centroid from {len(idxs)} samples")

w2v_test_preds = [
    max(unique_topics, key=lambda t: float(np.dot(emb, centroids_w2v[t])))
    for emb in test_embeddings_w2v
]

res_w2v = {
    'Accuracy':     accuracy_score(test_labels, w2v_test_preds),
    'Homogeneity':  homogeneity_score(test_labels, w2v_test_preds),
    'Completeness': completeness_score(test_labels, w2v_test_preds),
    'V-Measure':    v_measure_score(test_labels, w2v_test_preds)
}
print('Word2Vec Test Metrics:', res_w2v)


Training Word2Vec (50 epochs)...


Building W2V embeddings...


Building per-class prototype centroids...
  'Health' centroid from 305 samples
  'Politics' centroid from 305 samples
  'Sports' centroid from 305 samples
Word2Vec Test Metrics: {'Accuracy': 0.8964646464646465, 'Homogeneity': np.float64(0.6443682398310077), 'Completeness': np.float64(0.6461306944218478), 'V-Measure': np.float64(0.6452482636205192)}


In [5]:
# Word2Vec Embedding Inspection
print('\n--- Word2Vec Embedding Inspection ---')
print(f'Matrix Shape: {train_embeddings_w2v.shape} (Dense fixed-length representation)')
print('Sample W2V Vector (first 10 dims of sample 0):', train_embeddings_w2v[0][:10].round(4))
plot_clusters(test_embeddings_w2v, w2v_test_preds, 'Word2Vec Predicted Clusters (t-SNE)')



--- Word2Vec Embedding Inspection ---
Matrix Shape: (915, 100) (Dense fixed-length representation)
Sample W2V Vector (first 10 dims of sample 0): [ 0.2037  0.0141 -0.0802  0.1058 -0.0069  0.1125 -0.0091  0.0697 -0.1721
 -0.0175]


Saved: cluster_plot_Word2Vec_Predicted_Clusters_(t-SNE).png


In [6]:
# ── SBERT Baseline Pipeline ───────────────────────────────────────────────
# Uses the pre-trained SBERT K-Means model saved from the main pipeline.
model_path = "models/kmeans_model.pkl"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"{model_path} not found. Run main.py first.")

with open(model_path, "rb") as f:
    saved = pickle.load(f)
kmeans_sbert  = saved["model"]
mapping_sbert = saved["mapping"]

from sentence_transformers import SentenceTransformer
print("Loading SBERT model...")
sbert = SentenceTransformer("all-MiniLM-L6-v2")
test_embeddings_sbert = sbert.encode(test_texts, convert_to_numpy=True)

sbert_cluster_ids = kmeans_sbert.predict(test_embeddings_sbert)
preds_sbert = [mapping_sbert[cid] for cid in sbert_cluster_ids]

res_sbert = {
    "Accuracy":     accuracy_score(test_labels, preds_sbert),
    "Homogeneity":  homogeneity_score(test_labels, preds_sbert),
    "Completeness": completeness_score(test_labels, preds_sbert),
    "V-Measure":    v_measure_score(test_labels, preds_sbert)
}
print("SBERT Baseline Metrics:", res_sbert)


Loading SBERT model...


SBERT Baseline Metrics: {'Accuracy': 0.9722222222222222, 'Homogeneity': np.float64(0.8730720694963034), 'Completeness': np.float64(0.8731328890721458), 'V-Measure': np.float64(0.8731024782250647)}


In [7]:
# SBERT Embedding Inspection
print("\n--- SBERT Embedding Inspection ---")
print(f"Matrix Shape: {test_embeddings_sbert.shape} (Deep contextual sentence embeddings)")
print("Sample SBERT Vector (first 10 dims of sample 0):", test_embeddings_sbert[0][:10].round(4))

plot_clusters(test_embeddings_sbert, preds_sbert, "SBERT Predicted Clusters (t-SNE)")



--- SBERT Embedding Inspection ---
Matrix Shape: (396, 384) (Deep contextual sentence embeddings)
Sample SBERT Vector (first 10 dims of sample 0): [ 0.0159  0.0234  0.0506  0.0493  0.0222 -0.0025  0.0272 -0.0019  0.0115
 -0.0363]


Saved: cluster_plot_SBERT_Predicted_Clusters_(t-SNE).png


In [8]:
# Summary Comparison Table
results_df = pd.DataFrame({
    'TF-IDF':           res_tfidf,
    'Word2Vec (Scratch)': res_w2v,
    'SBERT (Baseline)': res_sbert,
}).T

print(results_df.round(4).to_string())

ax = results_df.plot(kind='bar', figsize=(10,5), colormap='Set2', rot=0)
ax.set_title('Model Comparison: TF-IDF vs Word2Vec vs SBERT')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison_bar.png', dpi=100)
plt.close()
print('Saved: model_comparison_bar.png')

os.makedirs('data', exist_ok=True)
results_df.to_csv('data/model_comparison_results.csv')
print('\nResults saved to data/model_comparison_results.csv')
print('\n' + '='*55)
print('       NOTEBOOK MODEL COMPARISON REPORT')
print('='*55)
for model, row in results_df.iterrows():
    print(f"  {model:<22}  Acc={row['Accuracy']:.4f}  H={row['Homogeneity']:.4f}  V={row['V-Measure']:.4f}")


                    Accuracy  Homogeneity  Completeness  V-Measure
TF-IDF                0.7753       0.6133        0.7123     0.6591
Word2Vec (Scratch)    0.8965       0.6444        0.6461     0.6452
SBERT (Baseline)      0.9722       0.8731        0.8731     0.8731


Saved: model_comparison_bar.png

Results saved to data/model_comparison_results.csv

       NOTEBOOK MODEL COMPARISON REPORT
  TF-IDF                  Acc=0.7753  H=0.6133  V=0.6591
  Word2Vec (Scratch)      Acc=0.8965  H=0.6444  V=0.6452
  SBERT (Baseline)        Acc=0.9722  H=0.8731  V=0.8731
